In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os
import time

# User Agent
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/76.0.3809.132 Safari/537.36'}

In [ ]:
# Extracting Flats/Apartments
flats = pd.DataFrame()

start = 1
end = 2
csv_file = f"/content/drive/MyDrive/DSMP/Case Studies/Real estate/flats_appartment/flats_gurgaon_data-p{start}-{end}.csv"

pageNumber = start
req=0

city = 'gurgaon'
i=1

while pageNumber < end:
    url = f'https://www.99acres.com/flats-in-{city}-ffid-page-{pageNumber}'
    page = requests.get(url, headers=headers)
    pageSoup = BeautifulSoup(page.content, 'html.parser')
    req+=1
    for soup in pageSoup.select_one('div[data-label="SEARCH"]').select('section[data-hydration-on-demand="true"]'):
        # Extract property name and property sub-name
        try:
            property_name = soup.select_one('a.srpTuple__propertyName').text.strip()
            # Extract link
            link = soup.select_one('a.srpTuple__propertyName')['href']
            society = soup.select_one('#srp_tuple_society_heading').text.strip()
        except:
            continue

        # Detail Page
        page = requests.get(link, headers=headers)
        dpageSoup = BeautifulSoup(page.content, 'html.parser')
        req += 1

        # price Range
        try:
            price = dpageSoup.select_one('#pdPrice2').text.strip()
        except:
            price = ''

        # Area
        try:
            area = soup.select_one('#srp_tuple_price_per_unit_area').text.strip()
        except:
            area = ''

        # Area with Type
        try:
            areaWithType = dpageSoup.select_one('#factArea').text.strip()
        except:
            areaWithType = ''
        
        # Configuration
        # bedRoom
        try:
            bedRoom = dpageSoup.select_one('#bedRoomNum').text.strip()
        except:
            bedRoom = ''

        # bathroom
        try:
            bathroom = dpageSoup.select_one('#bathroomNum').text.strip()
        except:
            bathroom = ''
        
        # balcony
        try:
            balcony = dpageSoup.select_one('#balconyNum').text.strip()
        except:
            balcony = ''

        # additionalRoom
        try:
            additionalRoom = dpageSoup.select_one('#additionalRooms').text.strip()
        except:
            additionalRoom = ''


        # Address
        try:
            address = dpageSoup.select_one('#address').text.strip()
        except:
            address = ''
        
        # floor Number
        try:
            floorNum = dpageSoup.select_one('#floorNumLabel').text.strip()
        except:
            floorNum = ''
        
        # facing
        try:
            facing = dpageSoup.select_one('#facingLabel').text.strip()
        except:
            facing = ''

        # agePossession
        try:
            agePossession = dpageSoup.select_one('#agePossessionLbl').text.strip()
        except:
            agePossession = ''

        # Nearby Locations
        try:
            nearbyLocations = [i.text.strip() for i in dpageSoup.select_one('div.NearByLocation__tagWrap').select('span.NearByLocation__infoText')]
        except:
            nearbyLocations = ''

        # Descriptions
        try:
            description = dpageSoup.select_one('#description').text.strip()
        except:
            description = ''

        # Furnish Details
        try:
            furnishDetails = [i.text.strip() for i in dpageSoup.select_one('#furnishDetails').select('li')]
        except:
            furnishDetails = ''
        
        # Features
        if furnishDetails:
            try:
                features = [i.text.strip() for i in dpageSoup.select('#features')[1].select('li')]
            except:
                features = ''
        else:
            try:
                features = [i.text.strip() for i in dpageSoup.select('#features')[0].select('li')]
            except:
                features = ''

        # Rating by Features
        try:
            rating = [i.text for i in dpageSoup.select_one('div.review__rightSide>div>ul>li>div').select('div.ratingByFeature__circleWrap')]
        except:
            rating = ''
        # print(top_f)

        # Property ID
        try:
            property_id = dpageSoup.select_one('#Prop_Id').text.strip()
        except:
            property_id = ''


        # Create a dictionary with the given variables
        property_data = {
            'property_name': property_name,
            'link': link,
            'society': society,
            'price': price,
            'area': area,
            'areaWithType': areaWithType,
            'bedRoom': bedRoom,
            'bathroom': bathroom,
            'balcony': balcony,
            'additionalRoom': additionalRoom,
            'address': address,
            'floorNum': floorNum,
            'facing': facing,
            'agePossession': agePossession,
            'nearbyLocations': nearbyLocations,
            'description': description,
            'furnishDetails': furnishDetails,
            'features': features,
            'rating': rating,
            'property_id': property_id
    }
        
        temp_df = pd.DataFrame.from_records([property_data])
        # print(temp_df)
        flats = pd.concat([flats, temp_df], ignore_index=True)
        i += 1
        if os.path.isfile(csv_file):
            # Append Dataframe to the existing file without header
            temp_df.to_csv(csv_file, mode='a', header=False, index=False)
        else:
            # Write Dataframe to the file with header
            temp_df.to_csv(csv_file, mode='a', header=True, index=False)


        if req % 4==0:
            time.sleep(10)
        if req % 15 == 0:
            time.sleep(50)

        print(f'{pageNumber} -> {i}')
        pageNumber += 1
    


In [ ]:
flats.info()

In [ ]:
flats

In [ ]:
def combine_csv_files(folder_path, combined_file_path):
    combined_data = pd.DataFrame() # Create an empty DataFrame to hold the combined data

    # Iterate through all CSV files in the folder
    for file_name in os.listdir(folder_path):
        if file_name.endswith('.csv'):
            file_path = os.path.join(folder_path, file_name)
            print(file_path)
            # Read the data from the current CSV file
            df = pd.read_csv(file_path)

            # Append the data to the combined DataFrame
            combined_data = combined_data.append(df, ignore_index=True)

            # Delete the original CSV file
            os.remove(file_path)

    # Save the combined data to a new CSV file
    combined_data.to_csv(combined_file_path, index=False)


# Example usage:

# Replace with the actual folder path
folder_path = '/content/drive/MyDrive/DSMP/Case Studies/Real estate/flats_appartment'

# Replace with the desired combined file path
combined_file_path = '/content/drive/MyDrive/DSMP/Case Studies/Real estate/flats_appartment/flats.csv'

combine_csv_files(folder_path, combined_file_path)

In [ ]:
pd.read_csv(combined_file_path)

<br>
<br>

### **-OWN DATASET**

In [7]:
%pip install faker

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.5/2.0 MB 929.6 kB/s eta 0:00:02
   ---------- ----------------------------- 0.5/2.0 MB 929.6 kB/s eta 0:00:02
   ---------------- ----------------------- 0.8/2.0 MB 657.8 kB/s eta 0:00:02
   ---------------- ----------------------- 0.8/2.0 MB 657.8 kB/s eta 0:00:02
   --------------------- ------------------ 1.0/2.0 MB 636.8 kB/s eta 0:00:02
   --------------------- ------------------ 1.0/2.0 MB 636.8 kB/s eta 0:00:02
   -------------------------- ------------- 1.3/2.0 MB 657.8 kB/s eta 0:00:01
   -------------------------- ------------- 1.3/2.0 MB 657.8 kB/s eta 0:00:01
   -------------------------------- ------- 1.6/2.0 MB 635.3 kB/s eta 0:00:01
   -------------------


[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import pandas as pd
from faker import Faker
import random
import uuid
from datetime import datetime, timedelta

# Initialize Faker for Indian context
fake = Faker('en_IN')

# Define lists of choices based on the provided images
area_types = ['Carpet', 'Super built up', 'Built up']
directions = ['North', 'South', 'East', 'West', 'North-East', 'South-East', 'North-West', 'South-West', 'Undefined']
age_ranges = ['0 to 1 Year Old', '1 to 5 Year Old', '5 to 10 Year Old', '10+ Year Old']
additional_rooms_list = ['Study Room', 'Servant Room', 'Pooja Room', 'Store Room', 'Not Available']
furnishings_list = ['Wardrobe', 'Fan', 'Light', 'AC', 'Geyser', 'Bed', 'Sofa', 'TV', 'Modular Kitchen']
features_list = ['Security / Fire Alarm', 'Intercom Facility', 'Lift(s)', 'Water purifier', 'Park', 'Swimming Pool', 'Fitness Centre / GYM', 'Club house / Community Center', 'Private Garden']

data = []

# Loop to generate 3000 rows of data
for _ in range(3000):
    # --- Basic Property Info ---
    num_bedrooms = random.randint(1, 5)
    sector = f"Sector {random.randint(50, 150)}"
    city = 'Gurgaon' # Focusing on Gurgaon as per the examples
    property_name = f"{num_bedrooms} BHK Flat in {sector}"

    # --- UPDATED LINK GENERATION ---
    # This now creates a real, working search URL
    actual_link = f"https://www.99acres.com/{num_bedrooms}-bhk-flats-in-{city.lower()}-ffid"
    
    # --- Price and Area ---
    price_in_lakhs = random.uniform(30.0, 900.0)
    price_str = f"{price_in_lakhs/100:.2f} Crore" if price_in_lakhs > 99 else f"{price_in_lakhs:.2f} Lac"
    
    area_sqft = random.randint(500, 5000)
    price_per_sqft = (price_in_lakhs * 100000) / area_sqft
    area_price_str = f"₹{int(price_per_sqft):,}/sq.ft."

    area_type = random.choice(area_types)
    area_with_type_str = f"{area_type} area: {area_sqft} sq.ft."

    # --- Room Details ---
    num_bathrooms = random.randint(1, num_bedrooms)
    num_balconies = random.randint(0, 3)
    additional_room = random.choices(additional_rooms_list, weights=[1, 1, 1, 1, 6], k=1)[0]
    
    # --- Location and Building Details ---
    society = fake.last_name() + " " + random.choice(["Residency", "Enclave", "Greens", "Homes"])
    address = f"{sector}, {city}"
    total_floors = random.randint(max(4, int(area_sqft/400)), 35)
    floor = random.randint(1, total_floors)
    floor_num_str = f"{floor}{'st' if floor==1 else 'nd' if floor==2 else 'rd' if floor==3 else 'th'} of {total_floors} Floors"

    # --- Possession and Other Features ---
    facing = random.choice(directions)
    
    if random.random() > 0.7:
        age_possession = "Under Construction"
        possession_date = datetime.now() + timedelta(days=random.randint(60, 1000))
        possession_str = possession_date.strftime('%b %Y') 
    else:
        age_possession = random.choice(age_ranges)
        possession_str = age_possession

    nearby_locations = str([fake.street_name()])
    
    # --- Context-Aware Description ---
    description = (
        f"{society} is one of {city}'s most sought after destinations for apartments. "
        f"This {num_bedrooms} bhk flat in {address} is your opportunity to be a part of this community. "
        f"The flat occupies a {area_type.lower()} area of {area_sqft} sq.ft. "
        f"It consists of {num_bedrooms} bedrooms and {num_bathrooms} bathrooms. "
        f"This flat is situated on the {floor_num_str}. "
        f"This {age_possession} property is available for possession from {possession_str}."
    )
    
    furnish_details = str(random.sample(furnishings_list, k=random.randint(2, 4))) if random.random() > 0.3 else '["Unfurnished"]'
    features = str(random.sample(features_list, k=random.randint(2, 5)))
    rating = f"{random.choice(['3', '4', '5'])} ({random.randint(5, 200)} reviews)"
    
    # --- Create Row Dictionary ---
    row = {
        'property_name': property_name,
        'link': actual_link, # Using the new, working link
        'society': society,
        'price': price_str,
        'area': area_price_str,
        'areaWithType': area_with_type_str,
        'bedRoom': f"{num_bedrooms} Bedrooms",
        'bathroom': f"{num_bathrooms} Bathrooms",
        'balcony': f"{num_balconies} {'Balconies' if num_balconies != 1 else 'Balcony'}",
        'additionalRoom': additional_room,
        'address': address,
        'floorNum': floor_num_str,
        'facing': facing,
        'agePossession': possession_str,
        'nearbyLocations': nearby_locations,
        'description': description,
        'furnishDetails': furnish_details,
        'features': features,
        'rating': rating,
        'property_id': f"S{random.randint(10000000, 99999999)}"
    }
    data.append(row)

# Create DataFrame and save to CSV
df = pd.DataFrame(data)
df.to_csv('flats.csv', index=False)

print("✅ Successfully generated 'flats.csv' with 3000 rows.")

✅ Successfully generated 'flats.csv' with 3000 rows.
